In [ ]:
import time
import pandas as pd
from fcapy.context import FormalContext
from fcapy.lattice import ConceptLattice
from fcapy.algorithms.concept_construction import close_by_one

from nextclosure import NextClosure
from algorithms.closebyone import CloseByOne

In [2]:
df_small = pd.read_csv("fca_benchmark_20x12.csv")
df_medium = pd.read_csv("fca_benchmark_40x20.csv")
df_large = pd.read_csv("fca_benchmark_100x50.csv")

In [3]:
def preprocess(df: pd.DataFrame):
    df = df.rename(columns={"Unnamed: 0": "object_id"})
    df = df.set_index("object_id")
    return df.astype(bool)

In [4]:
df_p_small = preprocess(df_small)
df_p_medium = preprocess(df_medium)
df_p_large = preprocess(df_large)

In [5]:
from dataclasses import dataclass

@dataclass
class TestResult:
    concepts: list
    total_time: float
    construction_time: float
    algorithm_time: float

In [6]:
def compare_nextclosure(df: pd.DataFrame):
    print(f"Data Shape: {df.shape}\n")
    print("Testing Library Implementation")
    c = time.perf_counter()
    K = FormalContext.from_pandas(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    # L = ConceptLattice.from_context(K)
    concepts_L = []
    for concept in close_by_one(K):
        concepts_L.append(concept)
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts_L)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    lib_results = TestResult(concepts=concepts_L, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()
    print("Testing self Implementation")
    c = time.perf_counter()
    algo = NextClosure()
    algo.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts = algo.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    self_results = TestResult(concepts=concepts, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()

    my_set = {
    (frozenset(objects), frozenset(attributes))
    for objects, attributes in concepts
    }

    lib_set = {
        (frozenset(c.extent), frozenset(c.intent))
        for c in concepts_L
    }
    print("Examining found concepts")
    if my_set == lib_set:
        print("--> Concepts are SAME across implementations.")
    else:
        print("--> Concepts are NOT same.")
        print(f"--> lib+/self- : {lib_set - my_set}")
        print(f"--> lib-/self+ : {my_set - lib_set}")

    return lib_results, self_results
    

In [9]:
def compare_cbo(df: pd.DataFrame):
    print(f"Data Shape: {df.shape}\n")
    print("Testing Library Implementation")
    c = time.perf_counter()
    K = FormalContext.from_pandas(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    # L = ConceptLattice.from_context(K)
    concepts_L = []
    for concept in close_by_one(K):
        concepts_L.append(concept)
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts_L)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    lib_results = TestResult(concepts=concepts_L, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()
    print("Testing self Implementation")
    c = time.perf_counter()
    algo = CloseByOne()
    algo.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts = algo.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    self_results = TestResult(concepts=concepts, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()

    my_set = {
    (frozenset(objects), frozenset(attributes))
    for objects, attributes in concepts
    }

    lib_set = {
        (frozenset(c.extent), frozenset(c.intent))
        for c in concepts_L
    }
    print("Examining found concepts")
    if my_set == lib_set:
        print("--> Concepts are SAME across implementations.")
    else:
        print("--> Concepts are NOT same.")
        print(f"--> lib+/self- : {lib_set - my_set}")
        print(f"--> lib-/self+ : {my_set - lib_set}")

    return lib_results, self_results
    

In [15]:
def compare_next_vs_cbo(df: pd.DataFrame):
    print(f"Data Shape: {df.shape}\n")
    print("Testing CbO Implementation")
    c = time.perf_counter()
    algo1 = NextClosure()
    algo1.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts1 = algo1.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts1)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    lib_results = TestResult(concepts=concepts1, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()
    print("Testing NextClosure Implementation")
    c = time.perf_counter()
    algo = CloseByOne()
    algo.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts = algo.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    self_results = TestResult(concepts=concepts, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()

    next_set = {
    (frozenset(objects), frozenset(attributes))
    for objects, attributes in concepts
    }

    cbo_set = {
        (frozenset(objects), frozenset(attributes))
        for objects, attributes in concepts
    }
    
    print("Examining found concepts")
    if next_set == cbo_set:
        print("--> Concepts are SAME across implementations.")
    else:
        print("--> Concepts are NOT same.")
        print(f"--> lib+/self- : {cbo_set - next_set}")
        print(f"--> lib-/self+ : {next_set - cbo_set}")

    return lib_results, self_results
    

In [7]:
lib_s, self_s = compare_nextclosure(df_p_small)
lib_m, self_m = compare_nextclosure(df_p_medium)
lib_l, self_l = compare_nextclosure(df_p_large)

Data Shape: (20, 12)

Testing Library Implementation
--> Concepts found: 88. Total time taken:  0.0237s. construction_pct = 1.00%

Testing self Implementation
--> Concepts found: 88. Total time taken:  0.0060s. construction_pct = 2.09%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (40, 20)

Testing Library Implementation
--> Concepts found: 546. Total time taken:  0.2694s. construction_pct = 0.13%

Testing self Implementation
--> Concepts found: 546. Total time taken:  0.0968s. construction_pct = 0.13%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (100, 50)

Testing Library Implementation
--> Concepts found: 32669. Total time taken:  24.3019s. construction_pct = 0.00%

Testing self Implementation
--> Concepts found: 32669. Total time taken:  10.7577s. construction_pct = 0.00%

Examining found concepts
--> Concepts are SAME across implementations.


In [11]:
libc_s, selfc_s = compare_cbo(df_p_small)
libc_m, selfc_m = compare_cbo(df_p_medium)
libc_l, selfc_l = compare_cbo(df_p_large)

Data Shape: (20, 12)

Testing Library Implementation
--> Concepts found: 88. Total time taken:  0.0277s. construction_pct = 0.96%

Testing self Implementation
--> Concepts found: 88. Total time taken:  0.0066s. construction_pct = 2.26%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (40, 20)

Testing Library Implementation
--> Concepts found: 546. Total time taken:  0.2882s. construction_pct = 0.09%

Testing self Implementation
--> Concepts found: 546. Total time taken:  0.1014s. construction_pct = 0.13%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (100, 50)

Testing Library Implementation
--> Concepts found: 32669. Total time taken:  26.6375s. construction_pct = 0.00%

Testing self Implementation
--> Concepts found: 32669. Total time taken:  11.9595s. construction_pct = 0.00%

Examining found concepts
--> Concepts are SAME across implementations.


In [16]:
selfn_s, selfc_s = compare_next_vs_cbo(df_p_small)
selfn_m, selfc_m = compare_next_vs_cbo(df_p_medium)
selfn_l, selfc_l = compare_next_vs_cbo(df_p_large)

Data Shape: (20, 12)

Testing CbO Implementation
--> Concepts found: 88. Total time taken:  0.0066s. construction_pct = 1.83%

Testing NextClosure Implementation
--> Concepts found: 88. Total time taken:  0.0046s. construction_pct = 3.02%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (40, 20)

Testing CbO Implementation
--> Concepts found: 546. Total time taken:  0.0976s. construction_pct = 0.12%

Testing NextClosure Implementation
--> Concepts found: 546. Total time taken:  0.1024s. construction_pct = 0.29%

Examining found concepts
--> Concepts are SAME across implementations.
Data Shape: (100, 50)

Testing CbO Implementation
--> Concepts found: 32669. Total time taken:  13.7947s. construction_pct = 0.00%

Testing NextClosure Implementation
--> Concepts found: 32669. Total time taken:  10.9930s. construction_pct = 0.00%

Examining found concepts
--> Concepts are SAME across implementations.


In [17]:
from fca.api_models import Context
import time

def benchmark_inclose(df):

    O = df.index.astype(str).tolist()
    A = df.columns.astype(str).tolist()
    I = df.to_numpy(dtype=int).tolist()

    # construction
    c = time.perf_counter()
    K = Context(O, A, I)
    ce = time.perf_counter()

    # In-Close
    s = time.perf_counter()
    concepts = K.get_concepts()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s

    print(
        f"--> Concepts found: {len(concepts)}. "
        f"Construction: {construction_time:.4f}s | "
        f"In-Close: {algorithm_time:.4f}s | "
        f"Total: {construction_time + algorithm_time:.4f}s"
    )

    return concepts

In [19]:
concepts = benchmark_inclose(df_p_small)
concepts = benchmark_inclose(df_p_medium)
concepts = benchmark_inclose(df_p_large)

--> Concepts found: 88. Construction: 0.0000s | In-Close: 0.0016s | Total: 0.0016s
--> Concepts found: 546. Construction: 0.0000s | In-Close: 0.0199s | Total: 0.0199s
--> Concepts found: 32669. Construction: 0.0000s | In-Close: 2.6409s | Total: 2.6409s


In [ ]:
def compare_next_vs_cbo(df: pd.DataFrame):
    print(f"Data Shape: {df.shape}\n")
    print("Testing CbO Implementation")
    c = time.perf_counter()
    algo1 = NextClosure()
    algo1.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts1 = algo1.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts1)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    lib_results = TestResult(concepts=concepts1, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()
    print("Testing NextClosure Implementation")
    c = time.perf_counter()
    algo = CloseByOne()
    algo.prepare(df)
    ce = time.perf_counter()

    s = time.perf_counter()
    concepts = algo.run()
    e = time.perf_counter()

    construction_time = ce - c
    algorithm_time = e - s
    total_time = construction_time + algorithm_time

    print(f"--> Concepts found: {len(concepts)}. Total time taken: {total_time: 0.4f}s. construction_pct = {construction_time/total_time * 100:0.2f}%")
    self_results = TestResult(concepts=concepts, total_time=total_time, construction_time=construction_time, algorithm_time=algorithm_time)
    print()

    next_set = {
    (frozenset(objects), frozenset(attributes))
    for objects, attributes in concepts
    }

    cbo_set = {
        (frozenset(objects), frozenset(attributes))
        for objects, attributes in concepts
    }
    
    print("Examining found concepts")
    if next_set == cbo_set:
        print("--> Concepts are SAME across implementations.")
    else:
        print("--> Concepts are NOT same.")
        print(f"--> lib+/self- : {cbo_set - next_set}")
        print(f"--> lib-/self+ : {next_set - cbo_set}")

    return lib_results, self_results
    